In [0]:
%run ./00_setup

In [0]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest

# 1. Cargar los datos
# Lee la tabla de clientes crudos desde Spark usando la variable SCHEMA
customers_df = spark.table(f"workspace.bronze.customers")

# 2. Inicializar el contexto de Great Expectations
# El 'context' es el punto de entrada principal para usar GX
context = gx.get_context()

# 3. Crear una "Suite de Expectativas" (Un conjunto de reglas)
suite_name = "customers_suite"
context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

# 4. Configurar la fuente de datos (Datasource) 
# Le decimos a GX que vamos a trabajar con Spark y le damos un nombre
datasource_config = {
    "name": "spark_customers_ds",
    "class_name": "Datasource",
    "execution_engine": {
        "class_name": "SparkDFExecutionEngine",
        "persist": False
    },
    "data_connectors": {
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
}

datasource = context.add_datasource(**datasource_config)

# 5. Crear el Validador usando RuntimeBatchRequest
# El validador es el motor que conecta los datos (el asset) con las reglas (la suite)
batch_request = RuntimeBatchRequest(
    datasource_name="spark_customers_ds",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="customers_asset",
    runtime_parameters={"batch_data": customers_df},
    batch_identifiers={"default_identifier_name": "customers_batch"}
)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# 6. Definir las Reglas de Calidad (Expectativas)

# Verifica que las columnas existan y estén exactamente en este orden
validator.expect_table_columns_to_match_ordered_list(
    column_list=["customer_id", "signup_date", "country", "segment", "email"]
)

# Verifica que el ID de cliente no tenga valores nulos (vacíos)
validator.expect_column_values_to_not_be_null("customer_id")

# Verifica que el ID de cliente sea único (sin duplicados)
validator.expect_column_values_to_be_unique("customer_id")

# Verifica que el país sea uno de los permitidos en la lista
validator.expect_column_values_to_be_in_set("country", ["ES", "MX", "AR", "CO", "CL"])

# Verifica que el segmento de negocio sea válido
validator.expect_column_values_to_be_in_set("segment", ["B2C", "SMB", "ENT"])

# Verifica que el email tenga un formato válido usando una Expresión Regular
# El parámetro mostly=0.99 permite un 1% de margen de error (ej. correos mal formados)
validator.expect_column_values_to_match_regex(
    "email",
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    mostly=0.90
)

# Verifica que la fecha de registro esté entre el 1 de enero de 2022 y el día de hoy
validator.expect_column_values_to_be_between(
    "signup_date",
    min_value="2022-01-01",
    max_value=str(dt.date.today())
)

# 7. Guardar y Ejecutar
# Guarda el conjunto de reglas en el contexto (no descarta las reglas que fallaron al probarlas)
validator.save_expectation_suite(discard_failed_expectations=False)

# Ejecuta la validación comparando los datos contra las reglas definidas
customers_result = validator.validate()

# 8. Mostrar y Guardar Resultados
# Muestra los resultados en formato JSON 
display(customers_result.to_json_dict())

# Guarda los resultados en la ruta especificada usando una función personalizada
save_json(
    f"{VALIDATION_PATH}/customers_validation.json",
    customers_result.to_json_dict()
)

In [0]:
import datetime as dt
import pandas as pd
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest

# 1. Cargar los datos con Pandas
# Puedes leerlo de un CSV, Excel, SQL o Parquet
customers_df = spark.table(f"workspace.bronze.customers")

customers_df = customers_df.toPandas()
# Asegúrate de que las fechas sean objetos datetime para que la validación funcione
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'])

In [0]:
customers_df.info()

In [0]:
# 2. Inicializar el contexto de Great Expectations
context = gx.get_context()

# 3. Crear o actualizar la Suite de Expectativas
suite_name = "customers_pandas_suite"
context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

# 4. Configurar la fuente de datos (Datasource) para PANDAS
# Cambiamos SparkDFExecutionEngine por PandasExecutionEngine
datasource_config = {
    "name": "pandas_customers_ds",
    "class_name": "Datasource",
    "execution_engine": {
        "class_name": "PandasExecutionEngine",
    },
    "data_connectors": {
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
}

context.add_datasource(**datasource_config)

# 5. Crear el Validador usando RuntimeBatchRequest para Pandas
batch_request = RuntimeBatchRequest(
    datasource_name="pandas_customers_ds",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="customers_asset_pandas",
    runtime_parameters={"batch_data": customers_df}, # Aquí pasamos el DataFrame de Pandas
    batch_identifiers={"default_identifier_name": "customers_batch_001"}
)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# 6. Definir las Reglas de Calidad (Expectativas)
# Las reglas son idénticas a las de Spark, GX se encarga de la traducción

validator.expect_table_columns_to_match_ordered_list(
    column_list=["customer_id", "signup_date", "country", "segment", "email"]
)

validator.expect_column_values_to_not_be_null("customer_id")
validator.expect_column_values_to_be_unique("customer_id")

validator.expect_column_values_to_be_in_set("country", ["ES", "MX", "AR", "CO", "CL"])
validator.expect_column_values_to_be_in_set("segment", ["B2C", "SMB", "ENT"])

validator.expect_column_values_to_match_regex(
    "email",
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    mostly=0.90
)

# Validación de fecha (Pandas maneja strings o objetos datetime)
validator.expect_column_values_to_be_between(
    "signup_date",
    min_value="2022-01-01",
    max_value=str(dt.date.today())
)

# 7. Guardar y Ejecutar
validator.save_expectation_suite(discard_failed_expectations=False)
customers_result = validator.validate()

# 8. Mostrar y Guardar Resultados
# Si estás en un notebook, esto imprimirá el JSON
print(customers_result)

# Para guardar (asumiendo que tienes definida la función save_json)
# save_json("customers_validation_pandas.json", customers_result.to_json_dict())

In [0]:
print(customers_result)